In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt #绘图工具
import seaborn as sns
import warnings
import time
import torch
warnings.filterwarnings("ignore")
#设置jupyter显示多行结果
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all' #默认为'last'
# #显示所有列
pd.set_option('display.max_columns', None) #原来中间会有部分列的显示被省略
# #显示所有行
# pd.set_option('display.max_rows', None)
#设置value的显示长度为100，默认为50
pd.set_option('max_colwidth',100)
plt.rcParams['font.sans-serif'] = ['SimHei']  #显示中文
plt.rcParams['axes.unicode_minus']=False #用来正常显示负号

In [2]:
rating  = np.loadtxt("./ml-100k/u.data", dtype=int, delimiter="\t")
rating[:5]

array([[      196,       242,         3, 881250949],
       [      186,       302,         3, 891717742],
       [       22,       377,         1, 878887116],
       [      244,        51,         2, 880606923],
       [      166,       346,         1, 886397596]])

In [3]:
rating_df = pd.DataFrame(rating)
rating_df.columns = ["user","item","rating","time"]
rating_df

,user,ite m,rating,time
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
...,...,...,...,...
99995,880,476,3,880175444
99996,716,204,5,879795543
99997,276,1090,1,874795795
99998,13,225,2,882399156


In [4]:
rating_df = rating_df.sort_values(by=["time"])
rating_df

,user,ite m,rating,time
214,259,255,4,874724710
83965,259,286,4,874724727
43027,259,298,4,874724754
21396,259,185,4,874724781
82655,259,173,4,874724843
...,...,...,...,...
46773,729,689,4,893286638
73008,729,313,3,893286638
46574,729,328,3,893286638
64312,729,748,4,893286638


In [5]:
rating_df["item"].nunique()
rating_df["item"].max()
rating_df["item"].min()

KeyError: 'item'

In [6]:
rating_df["user"].nunique()
rating_df["user"].max()
rating_df["user"].min()

943

943

1

In [7]:
rating_df["user"] = rating_df["user"].apply(lambda x: x-1)
rating_df["item"] = rating_df["item"].apply(lambda x: x-1)
rating_df["item"].nunique()
rating_df["item"].max()
rating_df["item"].min()
rating_df["user"].nunique()
rating_df["user"].max()
rating_df["user"].min()

1682

1681

0

943

942

0

In [8]:
rating_df["label"] = rating_df["rating"].apply(lambda x: 1 if x>3 else 0)

In [10]:
rating_df['time_stamp']=pd.to_datetime(rating_df['time'],unit='s',origin=pd.Timestamp('1970-01-01'))
rating_df

,user,item,rating,time,label,time_stamp
214,258,254,4,874724710,1,1997-09-20 03:05:10
83965,258,285,4,874724727,1,1997-09-20 03:05:27
43027,258,297,4,874724754,1,1997-09-20 03:05:54
21396,258,184,4,874724781,1,1997-09-20 03:06:21
82655,258,172,4,874724843,1,1997-09-20 03:07:23
...,...,...,...,...,...,...
46773,728,688,4,893286638,1,1998-04-22 23:10:38
73008,728,312,3,893286638,0,1998-04-22 23:10:38
46574,728,327,3,893286638,0,1998-04-22 23:10:38
64312,728,747,4,893286638,1,1998-04-22 23:10:38


In [11]:
rating_df.to_csv("ml-100k.csv",index=False, sep="\t")

In [9]:
rating_df_1 = rating_df[:50000].reset_index(drop=True)
rating_df_2 = rating_df[50000:].reset_index(drop=True)
rating_df_1
rating_df_2

,user,item,rating,time,label
0,258,254,4,874724710,1
1,258,285,4,874724727,1
2,258,297,4,874724754,1
3,258,184,4,874724781,1
4,258,172,4,874724843,1
...,...,...,...,...,...
49995,177,565,4,882826915,1
49996,177,650,4,882826915,1
49997,177,316,4,882826915,1
49998,177,194,4,882826944,1


,user,item,rating,time,label
0,177,97,5,882826944,1
1,177,678,4,882826944,1
2,177,384,4,882826982,1
3,177,317,5,882826982,1
4,177,133,3,882826983,0
...,...,...,...,...,...
49995,728,688,4,893286638,1
49996,728,312,3,893286638,0
49997,728,327,3,893286638,0
49998,728,747,4,893286638,1


In [10]:
rating_df_1[rating_df_1["rating"]>3].count()
rating_df_1[rating_df_1["rating"]<=3].count()
rating_df_1[rating_df_1["rating"]>3].count()/rating_df_1[rating_df_1["rating"]<=3].count()
print("--------------------")
rating_df_2[rating_df_2["rating"]>3].count()
rating_df_2[rating_df_2["rating"]<=3].count()
rating_df_2[rating_df_2["rating"]>3].count()/rating_df_2[rating_df_2["rating"]<=3].count()

user      28384
item      28384
rating    28384
time      28384
label     28384
dtype: int64

user      21616
item      21616
rating    21616
time      21616
label     21616
dtype: int64

user      1.313101
item      1.313101
rating    1.313101
time      1.313101
label     1.313101
dtype: float64

--------------------


user      26991
item      26991
rating    26991
time      26991
label     26991
dtype: int64

user      23009
item      23009
rating    23009
time      23009
label     23009
dtype: int64

user      1.173063
item      1.173063
rating    1.173063
time      1.173063
label     1.173063
dtype: float64

In [11]:
rating_df_1["item"].value_counts()
rating_df_2["item"].value_counts()

49      316
99      285
180     281
0       252
293     239
       ... 
1331      1
1319      1
1362      1
1311      1
538       1
Name: item, Length: 1466, dtype: int64

312     347
257     288
49      267
299     263
287     257
       ... 
1505      1
1634      1
1635      1
1637      1
1371      1
Name: item, Length: 1621, dtype: int64

In [12]:
item_1 = set(rating_df_1["item"])
len(item_1)
item_2 = set(rating_df_2["item"])
len(item_2)
len(item_1 & item_2)
len(item_2 - item_1)
# 100 - len(user_set[1] & user_set[2])


1466

1621

1405

216

In [13]:
user_1 = set(rating_df_1["user"])
len(user_1)
user_2 = set(rating_df_2["user"])
len(user_2)
len(user_1 & user_2)
len(user_2 - user_1)

491

586

134

452

In [14]:
online_learning = rating_df_1[["user","item","label"]]
batch_test = rating_df_2[["user","item","label"]]

In [16]:
from sklearn.utils import shuffle
online_learning_shuffle = shuffle(online_learning)
batch_test_shuffle = shuffle(batch_test)
online_learning_shuffle
batch_test_shuffle

,user,item,label
37215,373,126,1
12735,372,389,0
28636,458,650,0
29771,907,557,1
10125,406,858,0
...,...,...,...
30862,314,22,1
10221,600,672,0
12219,0,70,0
27364,150,142,1


,user,item,label
33240,777,27,1
11110,339,427,0
35178,930,274,1
9584,523,480,1
40442,658,1063,1
...,...,...,...
9828,716,306,1
28008,563,297,0
27771,141,180,1
32771,465,181,1


In [20]:
np.savetxt("./online_learning_8.txt", online_learning_shuffle[:40000].values, fmt="%d")
np.savetxt("./online_learning_2.txt", online_learning_shuffle[40000:].values, fmt="%d")
np.savetxt("./batch_test_8.txt", batch_test_shuffle[:40000].values, fmt="%d")
np.savetxt("./batch_test_2.txt", batch_test_shuffle[40000:].values, fmt="%d")

In [18]:
np.savetxt("./online_learning.txt", online_learning.values, fmt="%d")
np.savetxt("./batch_test.txt", batch_test.values, fmt="%d")

In [ ]:
# 两部分数据用MF分别学两个simulator，然后每次保留原始的log data以及随机采样99个作为候选集